In [0]:
# Databricks runtime already ships botocore/google-api-core — avoid -U on core deps.
# Pin urllib3/protobuf first, then install PA5 packages under the same constraints.
_CONSTRAINTS = "/tmp/pa5_pip_constraints.txt"
with open(_CONSTRAINTS, "w", encoding="utf-8") as _f:
    _f.write("urllib3>=1.25.4,<2.1\n")
    _f.write(
        "protobuf>=3.19.5,<5.0.0.dev0,!=3.20.0,!=3.20.1,"
        "!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5\n"
    )

%pip install -q --disable-pip-version-check -c {_CONSTRAINTS} "urllib3>=1.25.4,<2.1" "protobuf>=3.19.5,<5.0.0.dev0"
%pip install -q --disable-pip-version-check -c {_CONSTRAINTS} "typing_extensions>=4.10,<5" "tiktoken>=0.5" "pygame>=2.5" "mlflow>=2.9" "langchain-core>=0.3" "langchain>=0.3" "databricks-langchain>=0.1" "langgraph>=1.0.8,<1.1.0" "langgraph-prebuilt>=1.0.8,<1.0.9"

dbutils.library.restartPython()


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.69.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


**Databricks setup**
```python
mlflow.set_tracking_uri('databricks')
os.environ['ROLL_NUMBER'] = '25280091'  # output: 25280091_dino_runner.py
```

**Pip / environment:** The install cell pins `urllib3<2.1` and `protobuf<5` for Databricks. If you already ran the old `-U` install and see dependency warnings or broken imports, **detach and re-attach** the notebook (fresh cluster) before re-running.

**Checkpointing (Task 3.2):** Keep the same `PA5_THREAD_ID`. After a timeout during HITL `input()`, resume with `app.invoke(None, THREAD)` — do not restart from `initial_state`.

**Guardrail JSON path:** `/tmp/pa5_guardrail_report.json` on Databricks (not `~/Downloads` — that path changes per session).

**Task 5 deliverables (outside notebook):** zip `25280091_pa5.zip` with notebook, `pa5_guardrail_report.json`, game video link, MLflow screenshot (≥3 iteration runs).


In [0]:
"""PA5 — PA4 state keys + PA5 fields."""

from __future__ import annotations

import json
import os
import operator
import re
import subprocess
import sys
import tempfile
import time
import uuid
from typing import Annotated, List

import mlflow
import tiktoken
from typing_extensions import TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage
from langchain_core.tools import BaseTool
from pydantic import BaseModel, Field

from databricks_langchain import ChatDatabricks
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

_RUN_LOG_PREFIX = "__PA4_RUN_LOG__:\n"
MOSAIC_CHAT_ENDPOINT = (
    (os.environ.get("DATABRICKS_CHAT_ENDPOINT") or "").strip()
    or "databricks-meta-llama-3.1-405b-instruct"
)
MIN_ITERATIONS = 3
SUMMARY_TOKEN_THRESHOLD = int(os.getenv("PA5_SUMMARY_TOKEN_THRESHOLD", "2800"))
ENCODING = tiktoken.get_encoding("cl100k_base")
MLFLOW_EXPERIMENT = os.getenv("PA5_MLFLOW_EXPERIMENT", "/Shared/pa5_dino_runner")
# Databricks: use /tmp (stable). ~/Downloads path changes per cluster session.
GUARDRAIL_REPORT_PATH = os.getenv("PA5_GUARDRAIL_JSON", "/tmp/pa5_guardrail_report.json")


def _tok(text: str) -> int:
    return 0 if not text else len(ENCODING.encode(text))


def make_chat_model(temp_env: str, default_temp: float) -> ChatDatabricks:
    # Mosaic endpoints often cap max_tokens at 4096; do not exceed serving limit.
    _max_out = min(int(os.getenv("PA5_MAX_TOKENS", "4096")), 4096)
    return ChatDatabricks(
        endpoint=MOSAIC_CHAT_ENDPOINT,
        temperature=float(os.getenv(temp_env, str(default_temp))),
        max_tokens=_max_out,
    )


def architect_spec_from_state(state: dict) -> str:
    parts: List[str] = []
    for m in state.get("architect_messages") or []:
        if isinstance(m, AIMessage):
            parts.append(str(m.content))
    return "\n\n".join(parts).strip()


def messages_blob(msgs: List[BaseMessage]) -> str:
    return "\n".join(f"{type(m).__name__}: {getattr(m, 'content', m)!s}" for m in (msgs or []))


def estimate_history_tokens(state: dict) -> int:
    return (
        _tok(messages_blob(state.get("architect_messages") or []))
        + _tok(state.get("qa_feedback") or "")
        + _tok(state.get("architect_summary_note") or "")
    )


print("ChatDatabricks endpoint:", MOSAIC_CHAT_ENDPOINT)


ChatDatabricks endpoint: databricks-meta-llama-3.1-405b-instruct


In [0]:
class GameState(TypedDict):
    director_messages: Annotated[List[BaseMessage], add_messages]
    architect_messages: Annotated[List[BaseMessage], add_messages]
    engineer_code: str
    qa_feedback: str
    current_actor: str
    iteration: int
    iteration_score: Annotated[List[float], operator.add]
    file_saved: bool
    guardrail_report: dict
    architect_summary_note: str
    hitl_feedback_for_architect: str
    score_hitl_action: str
    summary_generation_count: int
    last_engineer_latency_sec: float
    last_qa_latency_sec: float


## PII filter (Director)


In [0]:
_EMAIL_RE = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", re.I)
_PASSWORD_FIELD_RE = re.compile(r"(?i)\b(password|passwd|pwd|secret)\s*[:=]\s*\S+")
_API_KEY_RE = re.compile(
    r"(?i)\b(api[_-]?key|access[_-]?token|bearer)\s*[:=]\s*[\w\-./+=]{8,}"
)
_GENERIC_KEY_RE = re.compile(r"\b(?:sk|rk|pk)_[a-zA-Z0-9]{16,}\b")


def redact_pii(text: str) -> tuple[str, dict]:
    report: dict = {"redactions": []}
    t = text
    if _EMAIL_RE.search(t):
        report["redactions"].append("email")
        t = _EMAIL_RE.sub("[REDACTED_EMAIL]", t)
    if _PASSWORD_FIELD_RE.search(t):
        report["redactions"].append("password_field")
        t = _PASSWORD_FIELD_RE.sub("[REDACTED_PASSWORD_FIELD]", t)
    if _API_KEY_RE.search(t):
        report["redactions"].append("api_credential")
        t = _API_KEY_RE.sub("[REDACTED_API_CREDENTIAL]", t)
    if _GENERIC_KEY_RE.search(t):
        report["redactions"].append("generic_key")
        t = _GENERIC_KEY_RE.sub("[REDACTED_KEY]", t)
    return t, report


def director_node(state: GameState):
    director_msg = input("Awaiting Director Prompt: ").strip()
    redacted, pii_ev = redact_pii(director_msg)
    gr = dict(state.get("guardrail_report") or {})
    gr["pii_at_director"] = {
        "blocked_or_redacted": bool(pii_ev["redactions"]),
        "patterns_matched": pii_ev["redactions"],
    }
    return {
        "director_messages": [
            HumanMessage(
                content=redacted,
                additional_kwargs={"role": "director", "pii_report": pii_ev},
            )
        ],
        "current_actor": "director",
        "guardrail_report": gr,
    }


def export_guardrail_report(state: dict | None = None) -> str:
    """Write pa5_guardrail_report.json to GUARDRAIL_REPORT_PATH (works on Databricks)."""
    gr = dict((state or {}).get("guardrail_report") or {})
    sample, pii_ev = redact_pii(
        "Contact dev@example.com password: secret123 api_key=sk_test_abcdefghijklmnop"
    )
    gr.setdefault(
        "pii_at_director",
        {
            "blocked_or_redacted": bool(pii_ev["redactions"]),
            "patterns_matched": pii_ev["redactions"],
            "redacted_preview": sample[:200],
        },
    )
    gr["pii_demo_redaction"] = {
        "blocked_or_redacted": True,
        "sample_input_redacted": sample,
        "note": "Task 5 JSON deliverable (deterministic demo)",
    }
    rep_path = GUARDRAIL_REPORT_PATH
    parent = os.path.dirname(rep_path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    with open(rep_path, "w", encoding="utf-8") as fp:
        fp.write(json.dumps(gr, indent=2))
    return rep_path


In [0]:
def architect_node(state: GameState):
    it = int(state.get("iteration", 0))
    print(f"\n========== ITERATION {it} (architect pass) ==========")
    brief = ""
    for m in reversed(state.get("director_messages") or []):
        if isinstance(m, HumanMessage) and m.additional_kwargs.get("role") == "director":
            brief = str(m.content)
            break
    override = (state.get("hitl_feedback_for_architect") or "").strip()
    qa_fb = (state.get("qa_feedback") or "").strip()
    if it > 0 and qa_fb and not qa_fb.startswith(_RUN_LOG_PREFIX):
        print("[QA feedback from last run]\n", qa_fb, sep="")
    if override:
        print("[HITL feedback for architect]\n", override, sep="")
    print(
        "Architect: Dino Runner — PA4 + PA5 (day/night, clustered obstacles, speed vs score, high score file).\n"
        f"Director brief: {brief!r}\n"
    )
    spec = input("Architect requirements (multi-line; empty line ends):\n")
    while True:
        line = input()
        if not line.strip():
            break
        spec += "\n" + line
    spec = spec.strip()
    if not spec and brief:
        spec = brief
    if override:
        spec = (override + "\n\n" + spec).strip()
    return {
        "architect_messages": [AIMessage(content=f"[Architect spec]\n{spec}")],
        "current_actor": "architect",
        "hitl_feedback_for_architect": "",
    }


## Summarization middleware


In [0]:
def summarization_middleware(state: GameState):
    it = int(state.get("iteration", 0))
    tot = estimate_history_tokens(state)
    # PDF: summarize when tokens exceed threshold; also run by iter 2+ to demo across 3+ cycles.
    demo_summarize = it >= MIN_ITERATIONS - 1
    if tot < SUMMARY_TOKEN_THRESHOLD and not demo_summarize:
        return {"current_actor": "summarization_skipped"}
    llm = make_chat_model("SUMMARY_TEMP", 0.2)
    arch_blob = messages_blob(state.get("architect_messages") or [])
    qa_piece = (state.get("qa_feedback") or "")[:6000]
    sys_m = SystemMessage(
        content=(
            "Summarize 'state of the game' for engineers (~400 words max): mechanics, obstacles, "
            "scoring, controls, PA5 features, open issues."
        )
    )
    human = f"## architect\n{arch_blob[:12000]}\n\n## qa\n{qa_piece}"
    ai = llm.invoke([sys_m, HumanMessage(content=human)])
    summary_text = (ai.content if hasattr(ai, "content") else str(ai)).strip()
    summary_msg = AIMessage(
        content=f"[SUMMARIZED_ARCHITECT_AND_QA]\n{summary_text}",
        additional_kwargs={"summarized_from_tokens": tot},
    )
    prev_n = int(state.get("summary_generation_count") or 0)
    return {
        "architect_messages": [summary_msg],
        "qa_feedback": "[SUMMARIZED — see architect_messages]",
        "architect_summary_note": summary_text[:200],
        "summary_generation_count": prev_n + 1,
        "current_actor": "summarization_middleware",
    }


## CodeInterpreterTool + Engineer / QA ReAct


In [0]:
class CodeInterpreterInput(BaseModel):
    code: str = Field(description="Python source to validate.")


class CodeInterpreterTool(BaseTool):
    name: str = "code_interpreter"
    description: str = "py_compile then ast.parse fallback on failure."
    args_schema: type[BaseModel] = CodeInterpreterInput

    def _fallback_ast(self, code: str) -> str:
        import ast

        try:
            ast.parse(code)
            return "[FALLBACK_OK] ast.parse OK"
        except SyntaxError as se:
            return f"[FALLBACK_SYNTAX_ERROR] {se}"

    def _run(self, code: str) -> str:
        try:
            with tempfile.NamedTemporaryFile("w", suffix="_pa5.py", delete=False, encoding="utf-8") as f:
                f.write(code)
                path = f.name
            exe = sys.executable or "python"
            chk = subprocess.run(
                [exe, "-m", "py_compile", path],
                capture_output=True,
                text=True,
                timeout=120,
            )
            msg = (chk.stdout or "") + (chk.stderr or "")
            try:
                os.unlink(path)
            except OSError:
                pass
            if chk.returncode != 0:
                return "PRIMARY_FAILED:\n" + msg + "\n---\n" + self._fallback_ast(code)
            return "PRIMARY_OK:\n" + msg
        except Exception as exc:
            return f"PRIMARY_EXCEPTION {exc!r}\n---\n" + self._fallback_ast(code)


code_interpreter_tool = CodeInterpreterTool()


def extract_py(src: str) -> str:
    if "```python" in src:
        blob = src.split("```python", 1)[1]
        return blob.split("```", 1)[0].strip()
    if "```" in src:
        return src.split("```", 1)[1].split("```", 1)[0].strip()
    return src.strip()


def engineer_react_node(state: GameState):
    """ReAct thought-action-observation loop (plain-text actions; avoids Databricks tool-JSON 400s)."""
    t0 = time.perf_counter()
    llm = make_chat_model("ENGINEER_TEMP", 0.2)
    spec = architect_spec_from_state(state)
    prev = (state.get("engineer_code") or "").strip()
    qa_fb = (state.get("qa_feedback") or "").strip()
    sys_txt = (
        "You are the Engineer in a ReAct loop. You may ONLY use plain text plus ```python fenced blocks. "
        "Do NOT use JSON tool calls. After each ```python block, the user message will give you the "
        "code_interpreter result (PRIMARY_OK or errors). Fix until PRIMARY_OK. "
        "When compilation succeeds, reply once more whose text contains the word FINAL and a single "
        "```python block with the COMPLETE game file. "
        "PA5: day/night, clustered obstacles, speed vs score, persistent high score; PA4: jump/duck, "
        "flying+ground obstacles, gravity."
    )
    chunks = [f"### Spec\n{spec or '(none)'}", f"### QA\n{qa_fb or '(none)'}"]
    if prev:
        chunks.append(f"### Prior code\n{prev[:80000]}")
    messages: List[BaseMessage] = [
        SystemMessage(content=sys_txt),
        HumanMessage(content="\n\n".join(chunks)),
    ]
    best_code = prev
    last_text = ""
    for step in range(10):
        ai = llm.invoke(messages)
        last_text = str(ai.content or "").strip()
        messages.append(AIMessage(content=last_text))
        block = extract_py(last_text)
        if not block:
            messages.append(
                HumanMessage(
                    content="No ```python block found. Output one fenced block for code_interpreter."
                )
            )
            continue
        obs = code_interpreter_tool.invoke({"code": block})
        if "PRIMARY_OK" in obs:
            best_code = block
        if "PRIMARY_OK" in obs and re.search(r"\bFINAL\b", last_text, re.I):
            break
        if step >= 7 and "PRIMARY_OK" in obs:
            break
        messages.append(
            HumanMessage(
                content=(
                    f"Observation (code_interpreter):\n{obs}\n\n"
                    "If not PRIMARY_OK, fix and send another ```python block. "
                    "If PRIMARY_OK and the game is complete, answer with FINAL and one full ```python file."
                )
            )
        )
    out_code = best_code or extract_py(last_text)
    dt = time.perf_counter() - t0
    return {
        "engineer_code": out_code,
        "current_actor": "engineer_react",
        "last_engineer_latency_sec": float(dt),
    }


class SpecComplianceInput(BaseModel):
    code: str = Field(description="Engineer Python source to check against PA4/PA5 spec.")
    spec: str = Field(description="Architect design text.")


class SpecComplianceTool(BaseTool):
    name: str = "spec_compliance_check"
    description: str = (
        "Deterministic scan: PA4 mechanics + PA5 day/night, clusters, speed vs score, highscore.json."
    )
    args_schema: type[BaseModel] = SpecComplianceInput

    _PA5_MARKERS = (
        ("day_night", re.compile(r"(day|night|DAY_COLOR|NIGHT_COLOR|toggle.*(day|night))", re.I)),
        ("clustered_obstacles", re.compile(r"(cluster|CLUSTER)", re.I)),
        ("speed_vs_score", re.compile(r"(speed.*score|score.*speed|increase.*speed)", re.I)),
        ("persistent_highscore", re.compile(r"(highscore|high_score|\.json)", re.I)),
        ("pa4_jump_duck", re.compile(r"(jump|duck|SPACE|DOWN|K_SPACE|K_DOWN)", re.I)),
        ("pa4_obstacles", re.compile(r"(obstacle|flying|ground|gravity)", re.I)),
    )

    def _run(self, code: str, spec: str = "") -> str:
        missing = [name for name, pat in self._PA5_MARKERS if not pat.search(code)]
        ok = [name for name, pat in self._PA5_MARKERS if pat.search(code)]
        return (
            f"SPEC_SCAN ok={ok} missing={missing} "
            f"code_len={len(code)} spec_len={len(spec)}"
        )


spec_compliance_tool = SpecComplianceTool()


def _parse_react_action(text: str) -> tuple[str, dict]:
    """Parse plain-text ReAct ACTION: tool / ARGS: json."""
    m = re.search(r"ACTION:\s*(\w+)", text, re.I)
    if not m:
        return "", {}
    tool = m.group(1).strip().lower()
    args_m = re.search(r"ARGS:\s*(\{.*?\})", text, re.S)
    args: dict = {}
    if args_m:
        try:
            args = json.loads(args_m.group(1))
        except json.JSONDecodeError:
            args = {}
    return tool, args


def qa_react_node(state: GameState):
    """QA ReAct: thought -> ACTION (tool) -> observation until FINAL QA report."""
    t0 = time.perf_counter()
    raw = (state.get("qa_feedback") or "").strip()
    run_out = raw[len(_RUN_LOG_PREFIX) :].strip() if raw.startswith(_RUN_LOG_PREFIX) else ""
    design = architect_spec_from_state(state).strip()
    code_txt = (state.get("engineer_code") or "").strip()
    llm = make_chat_model("QA_TEMP", 0.15)
    sys_txt = (
        "You are the QA Engineer in a ReAct loop (thought-action-observation). "
        "Use plain text only. To call a tool, emit:\n"
        "THOUGHT: ...\nACTION: spec_compliance_check | code_interpreter\n"
        'ARGS: {"code": "..."}  (add "spec" for spec_compliance_check)\n'
        "After observations, continue reasoning. When done, reply with FINAL QA REPORT "
        "and numbered findings + suggested fixes (no further ACTION lines)."
    )
    messages: List[BaseMessage] = [
        SystemMessage(content=sys_txt),
        HumanMessage(
            content=(
                f"### Architect spec\n{design[:8000]}\n\n"
                f"### Run logs\n{run_out[:8000]}\n\n"
                f"### Engineer code ({len(code_txt)} chars)\n{code_txt[:50000]}"
            )
        ),
    ]
    final_fb = ""
    last_text = ""
    for step in range(8):
        ai = llm.invoke(messages)
        last_text = str(ai.content or "").strip()
        messages.append(AIMessage(content=last_text))
        if re.search(r"\bFINAL\s+QA\s+REPORT\b", last_text, re.I):
            final_fb = last_text
            break
        tool, args = _parse_react_action(last_text)
        if not tool:
            messages.append(
                HumanMessage(
                    content=(
                        "No ACTION found. Either call spec_compliance_check or code_interpreter "
                        "with ARGS JSON, or finish with FINAL QA REPORT."
                    )
                )
            )
            continue
        if tool in ("spec_compliance_check", "spec_compliance", "spec_check"):
            obs = spec_compliance_tool.invoke(
                {"code": args.get("code") or code_txt, "spec": args.get("spec") or design}
            )
        elif tool in ("code_interpreter", "code_interpreter_tool"):
            obs = code_interpreter_tool.invoke({"code": args.get("code") or code_txt})
        else:
            obs = f"Unknown tool {tool!r}. Use spec_compliance_check or code_interpreter."
        messages.append(HumanMessage(content=f"Observation:\n{obs}"))
        if step >= 6 and not final_fb:
            final_fb = last_text + "\n\n[Truncated ReAct — last observation applied.]"
            break
    if not final_fb:
        final_fb = last_text or "QA produced no report."
    dt = time.perf_counter() - t0
    return {"qa_feedback": final_fb, "current_actor": "qa_react", "last_qa_latency_sec": float(dt)}


## PA4-style file + run (optional human run)


In [0]:
def _runner_filename() -> str:
    roll = (os.getenv("ROLL_NUMBER") or "").strip()
    return f"{roll}_dino_runner.py" if roll else "dino_runner.py"


def file_writer(state: GameState):
    code = (state.get("engineer_code") or "").strip() or "# empty\npass\n"
    path = os.path.abspath(_runner_filename())
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write(code)
    print((code[:500] + "\n...") if len(code) > 500 else code)
    print(f"\n[File saved: {path}]")
    return {"file_saved": True, "current_actor": "file_writer"}


def run_code(state: GameState):
    print("\n===== run_code (human) =====")
    fn = _runner_filename()
    if input(f"Run {fn}? (y/n): ").strip().lower() != "y":
        log = f"[skipped] user declined {fn}"
        return {"qa_feedback": _RUN_LOG_PREFIX + log, "current_actor": "run_code"}

    script = os.path.abspath(fn)
    exe = sys.executable or "python"
    chunks: List[str] = []
    chk = subprocess.run(
        [exe, "-m", "py_compile", script],
        capture_output=True,
        text=True,
        timeout=120,
        cwd=os.path.dirname(script) or ".",
    )
    chunks.append(f"py_compile exit {chk.returncode}\n{(chk.stdout or '') + (chk.stderr or '')}")
    if chk.returncode != 0:
        return {"qa_feedback": _RUN_LOG_PREFIX + "\n".join(chunks), "current_actor": "run_code"}
    try:
        proc = subprocess.run(
            [exe, script],
            capture_output=True,
            text=True,
            timeout=30,
            cwd=os.path.dirname(script) or ".",
        )
        chunks.append(f"run exit {proc.returncode}\n{(proc.stdout or '') + (proc.stderr or '')}")
    except subprocess.TimeoutExpired as exc:
        chunks.append("timeout 30s (pygame may not exit headless)")
        if exc.stdout:
            chunks.append(exc.stdout)
        if exc.stderr:
            chunks.append(exc.stderr)
    return {"qa_feedback": _RUN_LOG_PREFIX + "\n".join(chunks), "current_actor": "run_code"}


## Scorer + groundedness + MLflow per iteration


In [0]:
def score_node(state: GameState):
    spec = architect_spec_from_state(state).strip()
    eng = (state.get("engineer_code") or "").strip()
    qa_fb = (state.get("qa_feedback") or "").strip()
    judge = make_chat_model("SCORE_TEMP", 0.0)
    g_ai = judge.invoke(
        [
            SystemMessage(
                content=(
                    "Rate GROUNDEDNESS only: how well does the code implement the architect spec? "
                    "Reply with ONE number 0.0 to 1.0 only."
                )
            ),
            HumanMessage(content=f"SPEC:\n{spec[:12000]}\n\nCODE:\n{eng[:16000]}"),
        ]
    )
    gt = str(getattr(g_ai, "content", g_ai)).strip()
    gm = re.search(r"[-+]?\d*\.?\d+", gt)
    groundedness = float(gm.group(0)) if gm else 0.0
    groundedness = max(0.0, min(1.0, groundedness))

    s_ai = judge.invoke(
        [
            SystemMessage(
                content="From QA text, one score 1.0-10.0 (digits only).",
            ),
            HumanMessage(content=qa_fb[:16000] or "(no qa)"),
        ]
    )
    st = str(getattr(s_ai, "content", s_ai)).strip()
    sm = re.search(r"[-+]?\d*\.?\d+", st)
    score = float(sm.group(0)) if sm else 5.0
    score = max(1.0, min(10.0, score))

    prev_it = int(state.get("iteration", 0))
    new_it = prev_it + 1

    try:
        mlflow.set_experiment(MLFLOW_EXPERIMENT)
        with mlflow.start_run(run_name=f"pa5_iter_{new_it}_{uuid.uuid4().hex[:6]}"):
            mlflow.log_param("iteration", new_it)
            mlflow.log_metric("groundedness", groundedness)
            mlflow.log_metric("qa_score", score)
            mlflow.log_metric("engineer_latency_sec", float(state.get("last_engineer_latency_sec") or -1.0))
            mlflow.log_metric("qa_latency_sec", float(state.get("last_qa_latency_sec") or -1.0))
    except Exception as ex:
        print("[MLflow]", ex)

    return {
        "iteration_score": [score],
        "iteration": new_it,
        "current_actor": "scorer",
    }


## HITL gates between phases


In [0]:
def hitl_after_director(state: GameState):
    print("\n--- HITL after Director (pause before Architect) [Enter]=continue ---")
    input("> ")
    return {"current_actor": "hitl_director"}


def hitl_after_architect(state: GameState):
    print("\n--- HITL after Architect [Enter]=continue | r <text>=revise via architect ---")
    line = input("> ").strip()
    if line.lower().startswith("r"):
        fb = line[1:].strip() or input("Feedback: ").strip()
        return {"hitl_feedback_for_architect": fb, "current_actor": "hitl_arch"}
    return {"current_actor": "hitl_arch"}


def route_post_arch_hitl(state: GameState) -> str:
    return "architect" if (state.get("hitl_feedback_for_architect") or "").strip() else "summarize"


def hitl_after_engineer(state: GameState):
    input("\n--- HITL after Engineer [Enter] to write file ---")
    return {"current_actor": "hitl_engineer"}


def hitl_after_file_writer(state: GameState):
    print("\n--- HITL after File Writer [Enter] to run_code ---")
    input("> ")
    return {"current_actor": "hitl_file"}


def hitl_after_run(state: GameState):
    input("\n--- HITL after run_code [Enter] to QA ---")
    return {"current_actor": "hitl_run"}


def hitl_after_qa(state: GameState):
    input("\n--- HITL after QA [Enter] to score ---")
    return {"current_actor": "hitl_qa"}


def hitl_after_score(state: GameState):
    """Mandatory HITL after Scorer (PDF Task 2.2): quit, Architect revision, or engineer refine."""
    it = int(state.get("iteration", 0))
    scores = state.get("iteration_score") or []
    last = float(scores[-1]) if scores else float("nan")
    print(f"\n--- HITL after Scorer (Director) --- iteration={it} qa_score={last:.1f}/10")

    if it < MIN_ITERATIONS:
        print(f"PA5: minimum {MIN_ITERATIONS} iterations required (currently {it}).")
        input("Press Enter -> summarize -> engineer (summarization demo)...")
        return {"score_hitl_action": "refine", "current_actor": "hitl_score"}

    line = input(
        'Director: [q]=terminate (code OK) | [u <feedback>]=unacceptable -> Architect | '
        "[Enter]=refine engineer loop: "
    ).strip()
    if line.lower().startswith("q"):
        print("-> END")
        return {"score_hitl_action": "finish", "current_actor": "hitl_score"}
    if line.lower().startswith("u"):
        fb = line[1:].strip() or input("Director feedback for Architect: ").strip()
        print("-> Architect (revise planning)")
        return {
            "hitl_feedback_for_architect": fb,
            "score_hitl_action": "architect",
            "current_actor": "hitl_score",
        }
    print("-> summarize -> engineer")
    return {"score_hitl_action": "refine", "current_actor": "hitl_score"}


def route_post_score_hitl(state: GameState) -> str:
    action = (state.get("score_hitl_action") or "refine").strip()
    if action == "finish":
        return "finish"
    if action == "architect" or (state.get("hitl_feedback_for_architect") or "").strip():
        return "architect"
    return "summarize"


In [0]:
# Post-Scorer HITL: see hitl_after_score + route_post_score_hitl (PDF Task 2.2).


## LangGraph wiring + MemorySaver


In [0]:
checkpointer = MemorySaver()

workflow = StateGraph(GameState)
workflow.add_node("director", director_node)
workflow.add_node("hitl_dir", hitl_after_director)
workflow.add_node("architect", architect_node)
workflow.add_node("hitl_arch", hitl_after_architect)
workflow.add_node("summarize", summarization_middleware)
workflow.add_node("engineer", engineer_react_node)
workflow.add_node("hitl_eng", hitl_after_engineer)
workflow.add_node("file_writer", file_writer)
workflow.add_node("hitl_file", hitl_after_file_writer)
workflow.add_node("run_code", run_code)
workflow.add_node("hitl_run", hitl_after_run)
workflow.add_node("qa", qa_react_node)
workflow.add_node("hitl_qa", hitl_after_qa)
workflow.add_node("score", score_node)
workflow.add_node("hitl_score", hitl_after_score)

# PDF chain: each agent phase has HITL before the next phase.
workflow.add_edge(START, "director")
workflow.add_edge("director", "hitl_dir")
workflow.add_edge("hitl_dir", "architect")
workflow.add_edge("architect", "hitl_arch")
workflow.add_conditional_edges(
    "hitl_arch",
    route_post_arch_hitl,
    {"architect": "architect", "summarize": "summarize"},
)
workflow.add_edge("summarize", "engineer")
workflow.add_edge("engineer", "hitl_eng")
workflow.add_edge("hitl_eng", "file_writer")
workflow.add_edge("file_writer", "hitl_file")
workflow.add_edge("hitl_file", "run_code")
workflow.add_edge("run_code", "hitl_run")
workflow.add_edge("hitl_run", "qa")
workflow.add_edge("qa", "hitl_qa")
workflow.add_edge("hitl_qa", "score")
workflow.add_edge("score", "hitl_score")
workflow.add_conditional_edges(
    "hitl_score",
    route_post_score_hitl,
    {"architect": "architect", "summarize": "summarize", "finish": END},
)

app = workflow.compile(checkpointer=checkpointer)

THREAD = {"configurable": {"thread_id": os.getenv("PA5_THREAD_ID", "pa5-main-thread")}}
print("Compiled PA5 graph (PDF-compliant HITL + post-Scorer routing).")
print("Resume after timeout: app.invoke(None, THREAD)  # same PA5_THREAD_ID")


Compiled PA5 graph (PDF-compliant HITL + post-Scorer routing).
Resume after timeout: app.invoke(None, THREAD)  # same PA5_THREAD_ID


## Run + export guardrail JSON sample


In [0]:
os.environ.setdefault("ROLL_NUMBER", "25280091")
try:
    mlflow.set_tracking_uri("databricks")
except Exception:
    pass

initial_state: GameState = {
    "director_messages": [],
    "architect_messages": [],
    "engineer_code": "",
    "qa_feedback": "",
    "current_actor": "",
    "iteration": 0,
    "iteration_score": [],
    "file_saved": False,
    "guardrail_report": {},
    "architect_summary_note": "",
    "hitl_feedback_for_architect": "",
    "score_hitl_action": "",
    "summary_generation_count": 0,
    "last_engineer_latency_sec": 0.0,
    "last_qa_latency_sec": 0.0,
}

print("--- PA5 stream (ROLL_NUMBER=%s, thread=%s) ---" % (
    os.getenv("ROLL_NUMBER"),
    THREAD["configurable"]["thread_id"],
))
print("Tip: include PII once in Director prompt (email/api_key) for guardrail JSON deliverable.")
final = None
for ev in app.stream(initial_state, THREAD, stream_mode="values"):
    final = ev
    ca = ev.get("current_actor", "")
    if ca:
        print(f"… {ca}")

# Always write JSON (even if stream ended early); path is /tmp on Databricks.
path = export_guardrail_report(final if final else None)
print("Wrote", path)
print("Download: run the next cell, or %sh cat", path)


--- PA5 stream (ROLL_NUMBER=25280091, thread=pa5-main-thread) ---
Tip: include PII once in Director prompt (email/api_key) for guardrail JSON deliverable.


Awaiting Director Prompt:  Contact lead at student@university.edu password: MySecret99 api_key=sk_test_abcdefghijklmnopqrstuvwxyz Build a single-file pygame Chrome-style Dino Runner for PA5. Controls: SPACE jump, DOWN duck, ESC quit. PA4: ground + flying obstacles, gravity, collision game-over, HUD score. PA5: day/night every 100 points, ~30% clustered spawns (2-3 obstacles), speed increases with score, persistent high score in highscore.json in cwd. Window 800x400, pygame + stdlib only, if __name__ == "__main__".

… director

--- HITL after Director (pause before Architect) [Enter]=continue ---


>  

… hitl_director

========== ITERATION 0 (architect pass) ==========
Architect: Dino Runner — PA4 + PA5 (day/night, clustered obstacles, speed vs score, high score file).
Director brief: 'Contact lead at [REDACTED_EMAIL] [REDACTED_PASSWORD_FIELD] [REDACTED_API_CREDENTIAL] Build a single-file pygame Chrome-style Dino Runner for PA5. Controls: SPACE jump, DOWN duck, ESC quit. PA4: ground + flying obstacles, gravity, collision game-over, HUD score. PA5: day/night every 100 points, ~30% clustered spawns (2-3 obstacles), speed increases with score, persistent high score in highscore.json in cwd. Window 800x400, pygame + stdlib only, if __name__ == "__main__".'



...

… hitl_run
… qa_react



--- HITL after QA [Enter] to score --- 

… hitl_qa
… scorer

--- HITL after Scorer (Director) --- iteration=3 qa_score=4.0/10


Director: [q]=terminate (code OK) | [u <feedback>]=unacceptable -> Architect | [Enter]=refine engineer loop:  q

-> END
… hitl_score
Wrote /tmp/pa5_guardrail_report.json
Download: run the next cell, or %sh cat /tmp/pa5_guardrail_report.json
